
---

# 🟢 CHƯƠNG 1: NỀN TẢNG TOÁN HỌC & MẠNG NƠ-RON CƠ BẢN

---
### 1. Học Máy (Machine Learning) bản chất là gì?
Máy tính không "thông minh", nó chỉ là một cỗ máy dò dẫm tìm ra **Trọng số (Weights - $w$)** sao cho kết quả dự đoán khớp với thực tế nhất.
* **Công thức lõi:** $Prediction = w \cdot x$

### 2. Thuật toán Giảm độ dốc (Gradient Descent)
* **Sai số (Loss):** Đo lường xem máy đoán ngu cỡ nào. Hàm phổ biến: Mất mát bình phương (MSE) = $(Prediction - True\_Value)^2$.
* **Đạo hàm (Derivative):** Kim chỉ nam giúp máy tính biết nên tăng hay giảm trọng số $w$ để sai số nhỏ lại.
* **Cập nhật:** $w = w - (Learning\_Rate \cdot Derivative)$

### 3. Tính Phi Tuyến (Non-Linearity)
* Mọi sự phức tạp trên đời đều không phải là đường thẳng.
* **Hàm Kích Hoạt ReLU ($f(x) = \max(0, x)$):** Bẻ gãy các giá trị âm thành 0. Nó tạo ra các "nếp nhăn" cho bộ não AI, giúp AI học được các đường cong phức tạp. Không có ReLU, ghép 1000 lớp nơ-ron cũng chỉ bằng 1 lớp.

In [ ]:
# Mô phỏng AI học bằng Toán thuần túy (Không dùng thư viện)
x = 2.0; y_true = 6.0  # Tìm w sao cho w * 2 = 6
w = 1.0                # Đoán bừa w ban đầu
lr = 0.05              # Tốc độ học

print("--- CHƯƠNG 1: GRADIENT DESCENT THỦ CÔNG ---")
for step in range(10):
    prediction = w * x
    error = prediction - y_true
    loss = error ** 2
    derivative = 2 * error * x    # Đạo hàm
    w = w - lr * derivative       # Cập nhật trọng số
    
    if step % 2 == 0:
        print(f"Bước {step}: Trọng số w = {w:.3f} | Sai số: {loss:.3f}")
print(f"✅ AI đã tìm ra quy luật: w = {w:.0f}")

---

# 🔵 CHƯƠNG 2: BƯỚC TIẾN LÊN DEEP LEARNING VỚI PYTORCH

---
Việc tính đạo hàm bằng tay chỉ dành cho đồ chơi. Khi AI có hàng tỷ tham số (như GPT-4), con người không thể giải tích nổi.
* **Tại sao cần PyTorch?**
  1. **Tensors:** Chạy ma trận trên GPU siêu tốc.
  2. **Autograd:** Tự động tính toán MỌI đạo hàm trên đời chỉ với lệnh `loss.backward()`.
* **Bộ 3 Phép màu của Huấn luyện:**

  1. `optimizer.zero_grad()`: Xóa sạch bộ nhớ đạo hàm cũ.
  2. `loss.backward()`: Kích hoạt Autograd tính đạo hàm mới.
  3. `optimizer.step()`: Trừ đạo hàm vào Trọng số (Giống hệt vòng lặp thủ công ở trên).

In [ ]:
import torch
import torch.nn as nn

print("\n--- CHƯƠNG 2: PYTORCH AUTOGRAD ---")
# Dữ liệu dạng Tensor
X = torch.tensor([[3.0, 10.0]]) 
Y = torch.tensor([[500.0]])

# Mạng Nơ-ron Đa Tầng (Có ReLU ở giữa)
model = nn.Sequential(
    nn.Linear(2, 4),
    nn.ReLU(),
    nn.Linear(4, 1)
)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.1)

for step in range(50):
    optimizer.zero_grad()            # 1. Reset
    pred = model(X)                  # 2. Đoán
    loss = criterion(pred, Y)        # 3. Tính sai số
    loss.backward()                  # 4. Tự động tính đạo hàm
    optimizer.step()                 # 5. Cập nhật

print(f"✅ PyTorch đã train xong! Dự đoán cuối: {model(X).item():.1f} (Giá thật: 500)")

---

# 🟣 CHƯƠNG 3.1: KIẾN TRÚC TRANSFORMER & CHATGPT

---
Làm sao AI hiểu được ngôn ngữ con người?
#### Bước 1: Mã hóa Từ vựng (Token Embeddings)
* Dịch chữ thành ID số (Tra từ điển).
* Ép ID số thành **Vector Không Gian** (`nn.Embedding`). Các từ cùng nghĩa sẽ có vector nằm gần nhau.

#### Bước 2: Mã hóa Vị trí (Positional Encoding)
* Mạng Transformer đọc tất cả các từ **cùng 1 lúc** (song song), nên nó bị "mù" thứ tự (không biết từ nào đứng trước).
* Phải cộng thêm một vector mang "số báo danh vị trí" vào vector từ vựng.
* $Input = Vector\_Ngữ\_Nghĩa + Vector\_Vị\_Trí$

#### Bước 3: Cơ chế Self-Attention (Tự Chú Ý - Trái tim của AI)
* Giúp các từ "nói chuyện" với nhau để tìm ngữ cảnh (vd: từ "Nó" là con mèo hay con chuột).
* Từng từ sinh ra 3 bản sao:
  - **Query (Q):** Tôi đang tìm kiếm điều gì?
  - **Key (K):** Tôi có nhãn dán/đặc tính gì?
  - **Value (V):** Giá trị ngữ nghĩa thực sự của tôi.
* **Công thức cốt lõi:** Nhân Q với K (Chuyển vị) để xem độ khớp. Biến thành % bằng Softmax. Sau đó nhân lấy thông tin từ V.

In [ ]:
import math
import torch
import torch.nn as nn

print("\n--- CHƯƠNG 3: SELF-ATTENTION TRONG TRANSFORMER ---")
# Giả sử ta có 3 từ, mỗi từ là vector 4 chiều. Đã được cộng Positional Encoding.
# Kích thước x: [1 câu, 3 từ, vector 4 chiều]
x = torch.randn(1, 3, 4) 

d_model = 4
# Tạo 3 Lớp sinh ra Q, K, V
W_q = nn.Linear(d_model, d_model, bias=False)
W_k = nn.Linear(d_model, d_model, bias=False)
W_v = nn.Linear(d_model, d_model, bias=False)

Q = W_q(x)
K = W_k(x)
V = W_v(x)

# Ma thuật Attention: Q nhân với K chuyển vị (-2, -1)
scores = Q @ K.transpose(-2, -1) / math.sqrt(d_model)

# Softmax biến điểm số thành % sự tập trung
attention_weights = torch.softmax(scores, dim=-1)

# Nhân % tập trung với Value thực tế của các từ
output = attention_weights @ V

print("Hình dáng đầu ra (Shape):", output.shape)
print("✅ Vector của mỗi từ hiện đã hòa quyện với ngữ cảnh của các từ xung quanh!")


--- CHƯƠNG 3: SELF-ATTENTION TRONG TRANSFORMER ---
Hình dáng đầu ra (Shape): torch.Size([1, 3, 4])
✅ Vector của mỗi từ hiện đã hòa quyện với ngữ cảnh của các từ xung quanh!
================ CHÚC MỪNG BẠN ĐÃ TỐT NGHIỆP KIẾN THỨC NỀN! ================


---

# 🔠 CHƯƠNG 3.2: MÃ HÓA TỪ VỰNG & NHẬN THỨC VỊ TRÍ (EMBEDDINGS)

---
Máy tính chỉ hiểu những con số. Để AI đọc hiểu tiếng người, ta phải dịch chữ thành số theo 2 bước:

### 1. Token Embeddings (Không gian Ngữ nghĩa)
* **Sai lầm thường gặp:** Gán 1 con số cho 1 từ (Apple=1, Dog=2). Máy sẽ nghĩ Dog lớn gấp đôi Apple (Sai logic toán học).
* **Giải pháp:** Chuyển mỗi từ thành một **Vector** (Một dãy gồm nhiều con số). Những từ cùng nghĩa (Vd: Vua, Hoàng Hậu) sẽ có các con số gần giống nhau.
* Lớp `nn.Embedding` của PyTorch chính là một cuốn từ điển khổng lồ để tra cứu các vector này.

### 2. Positional Encoding (Nhận thức Thứ tự)
* **Vấn đề:** Trái tim của AI (Transformer) đọc TẤT CẢ các từ trong câu cùng một lúc để chạy cho nhanh. Nhưng vì thế, nó bị "mù", không biết từ nào đứng trước, từ nào đứng sau.
* **Giải pháp:** Tạo ra một "Vector Vị Trí" mang số báo danh (0, 1, 2...) và **cộng trực tiếp** nó vào Vector Ngữ nghĩa ở trên.
* **Công thức lõi:** $Input = Vector\_Ngữ\_Nghĩa + Vector\_Vị\_Trí$


In [ ]:
import torch
import torch.nn as nn

print("--- CHƯƠNG 3: CHUẨN BỊ ĐẦU VÀO CHO AI ---")

# Giả sử ta có câu: "I love AI"
# Từ điển (Vocab): "I"=0, "love"=1, "AI"=2
input_ids = torch.tensor([0, 1, 2])
seq_len = len(input_ids) # Độ dài câu: 3 từ
d_model = 4 # Kích thước vector: Mỗi từ sẽ được miêu tả bằng 4 con số

print(f"1. ID Từ vựng (Token IDs): {input_ids.tolist()}")

# ==========================================
# 1. NHÚNG TỪ VỰNG (TOKEN EMBEDDING)
# ==========================================
# Khởi tạo lớp nhúng: Từ điển có 10 từ, mỗi từ thành vector 4 chiều
token_embedder = nn.Embedding(num_embeddings=10, embedding_dim=d_model)
word_vectors = token_embedder(input_ids)

print("\n2. Vector Ngữ Nghĩa (Word Vectors):")
print(word_vectors)

# ==========================================
# 2. NHÚNG VỊ TRÍ (POSITIONAL ENCODING)
# ==========================================
# Tạo ID vị trí: [0, 1, 2]
position_ids = torch.arange(0, seq_len)
print(f"\n3. ID Vị trí (Position IDs): {position_ids.tolist()}")

pos_embedder = nn.Embedding(num_embeddings=10, embedding_dim=d_model)
position_vectors = pos_embedder(position_ids)

# ==========================================
# 3. KẾT HỢP: Ý NGHĨA + THỨ TỰ
# ==========================================
# Phép cộng ma thuật của Deep Learning!
final_input = word_vectors + position_vectors

print("\n4. ĐẦU VÀO CUỐI CÙNG CHO TRANSFORMER (Đã cộng Vị trí):")
print(f"Hình dáng (Shape): {final_input.shape} -> (3 từ, mỗi từ 4 chiều)")
print(final_input)

---

# 🎭 CHƯƠNG 4: MẶT NẠ TIÊN TRI (CAUSAL MASK) - BÍ MẬT CỦA GPT

---

### 🔮 1. Vấn đề "Nhìn trộm tương lai"
* Kiến trúc Transformer gốc (của Google) cho phép tất cả các từ nhìn thấy nhau (cả quá khứ lẫn tương lai) để **Dịch thuật** cho chuẩn.
* Nhưng **GPT** (của OpenAI) lại sinh ra để **Sáng tác/Dự đoán** từ tiếp theo.
* Nếu đang đứng ở từ số 1 mà AI lại nhìn thấy trước từ số 2 và 3, nó sẽ "gian lận" và không học được cách suy luận.

### 🔺 2. Giải pháp: Ma trận Tam giác dưới (Lower Triangular Matrix)
* Các kỹ sư AI tạo ra một "mặt nạ" hình tam giác:
  - **Số 1 (Bật):** Chỉ cho phép nhìn chính mình và các từ đứng trước (Quá khứ).
  - **Số 0 (Tắt):** Bịt mắt hoàn toàn các từ đứng sau (Tương lai).

### 🧮 3. Tại sao lại thay số 0 bằng Âm Vô Cùng (-inf)?
* Trong toán học của mạng Nơ-ron, điểm số Attention sẽ được đưa qua hàm **Softmax** để biến thành %.
* Đặc tính kỳ diệu của Softmax: $Softmax(-\infty) = 0\%$.
* Vậy nên ta thay các số 0 trên mặt nạ thành $-\infty$, để ép AI dành chính xác **0% sự tập trung** cho tương lai!

In [2]:
import torch
import torch
import torch.nn as nn

print("--- CHƯƠNG 4: ĐEO MẶT NẠ CHO MA TRẬN ATTENTION ---")

# 1. BẢNG ĐIỂM ATTENTION THÔ (Ví dụ câu 3 từ: "Tôi", "thích", "AI")
# Kích thước: [3 từ đang chấm điểm chéo cho 3 từ]
raw_scores = torch.tensor([
    [10.0, 20.0, 30.0],  # Từ 1 "Tôi" đang nhìn thấy cả điểm của tương lai (20, 30)
    [15.0, 25.0, 10.0],  
    [12.0, 18.0, 20.0]   
])

print("\n1. Bảng điểm ban đầu (Gian lận nhìn trộm tương lai):")
print(raw_scores)

# 2. TẠO MẶT NẠ (Hàm torch.tril tạo ma trận tam giác dưới)
mask = torch.tril(torch.ones(3, 3))

print("\n2. Hình dáng mặt nạ (1: Được nhìn | 0: Bịt mắt):")
print(mask)

# 3. ĐEO MẶT NẠ (Chỗ nào là 0 thì đè float('-inf') vào)
masked_scores = raw_scores.masked_fill(mask == 0, float('-inf'))

print("\n3. Bảng điểm sau khi đeo mặt nạ (-inf = Bịt mắt):")
print(masked_scores)

# 4. CHUYỂN THÀNH XÁC SUẤT BẰNG SOFTMAX
attention_weights = torch.softmax(masked_scores, dim=-1)

print("\n4. Kết quả % sự tập trung (Softmax):")
print(attention_weights)
print("=> Dòng 1 (Từ 'Tôi') giờ chỉ tập trung 100% (1.0) vào bản thân nó. Không còn bị lộ đáp án tương lai!")

--- CHƯƠNG 4: ĐEO MẶT NẠ CHO MA TRẬN ATTENTION ---

1. Bảng điểm ban đầu (Gian lận nhìn trộm tương lai):
tensor([[10., 20., 30.],
        [15., 25., 10.],
        [12., 18., 20.]])

2. Hình dáng mặt nạ (1: Được nhìn | 0: Bịt mắt):
tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])

3. Bảng điểm sau khi đeo mặt nạ (-inf = Bịt mắt):
tensor([[10., -inf, -inf],
        [15., 25., -inf],
        [12., 18., 20.]])

4. Kết quả % sự tập trung (Softmax):
tensor([[1.0000e+00, 0.0000e+00, 0.0000e+00],
        [4.5398e-05, 9.9995e-01, 0.0000e+00],
        [2.9539e-04, 1.1917e-01, 8.8054e-01]])
=> Dòng 1 (Từ 'Tôi') giờ chỉ tập trung 100% (1.0) vào bản thân nó. Không còn bị lộ đáp án tương lai!


---

# 🥞 CHƯƠNG 5: MẠNG SUY NGHĨ SÂU & CHUẨN HÓA (FEED-FORWARD & LAYER NORM)

---

### ⚖️ 1. Người giữ thăng bằng (Layer Normalization)
* Sau nhiều phép nhân ma trận ở Self-Attention, các con số có thể bị cộng dồn lên quá to (ví dụ: 100.0, 500.0) hoặc ép xuống quá nhỏ.
* Nếu để nguyên, mạng nơ-ron sẽ bị "đột quỵ" (lỗi Gradient Exploding/Vanishing).
* **Layer Normalization** là một "chiếc van an toàn". Nó tự động tính trung bình cộng và ép các con số của mỗi từ về lại trạng thái ổn định (thường xoay quanh số 0, từ -1 đến 1).

### 🧠 2. Mạng Truyền Thẳng (Feed-Forward Network - FFN)
* Đây là không gian riêng để mỗi từ tự "tiêu hóa" ngữ cảnh thu được sau buổi họp nhóm.
* Cấu trúc siêu quen thuộc (Trò đã code ở Day 32): `Linear -> ReLU -> Linear`.
* **Quy tắc 4X của Transformer:** 1. Phóng to vector lên gấp 4 lần để "nhìn" vấn đề ở nhiều góc độ (vd: 64 chiều -> 256 chiều).
  2. Đi qua hàm kích hoạt **ReLU** để loại bỏ các thông tin rác (bẻ số âm thành 0) và giữ lại những đặc trưng sâu sắc nhất.
  3. Thu nhỏ vector về lại kích thước ban đầu (256 chiều -> 64 chiều) để chuẩn bị cho các khối tiếp theo.

In [3]:
import torch
import torch.nn as nn

print("--- CHƯƠNG 5: CHUẨN HÓA VÀ SUY NGHĨ SÂU ---")

# Giả sử ta có vector của từ "Tôi" sau khi đã "họp nhóm" (Self-Attention) xong
# Kích thước: [1 câu, 1 từ, vector 4 chiều]
d_model = 4
x_after_attention = torch.tensor([[[15.0, -20.0, 100.0, -5.0]]])

print("\n1. ĐẦU VÀO SAU KHI HỌP NHÓM (ATTENTION):")
print("Vector thô, các con số nhảy múa lộn xộn (có số rất to là 100.0):")
print(x_after_attention)

# ==========================================
# CHUẨN HÓA LỚP (LAYER NORMALIZATION)
# ==========================================
norm_layer = nn.LayerNorm(d_model)
x_normalized = norm_layer(x_after_attention)

print("\n2. SAU KHI CHUẨN HÓA (LAYER NORM):")
print("Các con số đã được ép về dạng cân bằng (mượt mà và an toàn hơn):")
print(x_normalized)

# ==========================================
# MẠNG SUY NGHĨ SÂU (FEED-FORWARD NETWORK)
# ==========================================
# Phóng to x4 -> ReLU -> Thu nhỏ
ffn = nn.Sequential(
    nn.Linear(d_model, d_model * 4), # Phóng to góc nhìn (4 -> 16)
    nn.ReLU(),                       # Bẻ gãy tuyến tính (Tạo nếp nhăn não)
    nn.Linear(d_model * 4, d_model)  # Rút ra kết luận (16 -> 4)
)

final_output = ffn(x_normalized)

print("\n3. ĐẦU RA CUỐI CÙNG (SAU FFN):")
print(f"Hình dáng: {final_output.shape}")
print(final_output)
print("=> ✅ TỪ NÀY ĐÃ TỰ TIÊU HÓA XONG NGỮ CẢNH VÀ ĐƯA RA ĐƯỢC ĐẶC TRƯNG SÂU SẮC NHẤT!")


--- CHƯƠNG 5: CHUẨN HÓA VÀ SUY NGHĨ SÂU ---

1. ĐẦU VÀO SAU KHI HỌP NHÓM (ATTENTION):
Vector thô, các con số nhảy múa lộn xộn (có số rất to là 100.0):
tensor([[[ 15., -20., 100.,  -5.]]])

2. SAU KHI CHUẨN HÓA (LAYER NORM):
Các con số đã được ép về dạng cân bằng (mượt mà và an toàn hơn):
tensor([[[-0.1615, -0.9152,  1.6690, -0.5922]]],
       grad_fn=<NativeLayerNormBackward0>)

3. ĐẦU RA CUỐI CÙNG (SAU FFN):
Hình dáng: torch.Size([1, 1, 4])
tensor([[[ 0.4232, -0.1379,  0.2213,  0.5326]]], grad_fn=<ViewBackward0>)
=> ✅ TỪ NÀY ĐÃ TỰ TIÊU HÓA XONG NGỮ CẢNH VÀ ĐƯA RA ĐƯỢC ĐẶC TRƯNG SÂU SẮC NHẤT!


---

# 🧱 CHƯƠNG 6: LẮP RÁP BỘ NÃO (TRANSFORMER BLOCK & RESIDUAL CONNECTION)

---

### 1. Đường tắt - Cộng phần dư (Residual Connection)
* **Vấn đề:** Khi đi qua quá nhiều lớp xử lý (Attention, FFN), dữ liệu gốc có thể bị biến dạng hoặc mất mát hoàn toàn.
* **Giải pháp:** Sau khi dữ liệu đi qua một bước xử lý, ta lấy kết quả đó **CỘNG TRỰC TIẾP** với dữ liệu gốc ban đầu.
* **Công thức cốt lõi:** $X_{mới} = X_{gốc} + Lớp\_Xử\_Lý(X_{gốc})$

### 2. Khối Transformer (Transformer Block)
Sức mạnh của GPT không đến từ một thuật toán ma thuật nào khác, mà đến từ việc thực hiện lặp đi lặp lại 4 trạm kiểm soát sau:
1. `LayerNorm` (Chuẩn hóa)
2. `Self-Attention` (Họp nhóm) + **Đường tắt (Residual)**
3. `LayerNorm` (Chuẩn hóa)
4. `Feed-Forward` (Suy nghĩ riêng) + **Đường tắt (Residual)**

In [8]:
import torch
import torch.nn as nn

print("--- CHƯƠNG 6: DÒNG CHẢY QUA KHỐI TRANSFORMER ---")

d_model = 4
# Giả sử ta có 1 câu, 3 từ, mỗi từ 4 chiều (Đã nhúng từ và vị trí)
x = torch.randn(1, 3, d_model)

print("1. Hình dáng Đầu vào ban đầu:", x.shape)

# ==========================================
# CHUẨN BỊ 4 TRẠM KIỂM SOÁT (LAYERS)
# ==========================================
# Trạm 1 & 3: Chuẩn hóa để giữ thăng bằng
ln1 = nn.LayerNorm(d_model)
ln2 = nn.LayerNorm(d_model)

# Trạm 2: Họp nhóm (Self-Attention)
attention = nn.MultiheadAttention(embed_dim=d_model, num_heads=1, batch_first=True)

# Trạm 4: Suy nghĩ sâu (Feed-Forward)
ffn = nn.Sequential(
    nn.Linear(d_model, d_model * 4),
    nn.ReLU(),
    nn.Linear(d_model * 4, d_model)
)

# ==========================================
# BẮT ĐẦU CHẠY DÒNG DỮ LIỆU (DATA FLOW)
# ==========================================

# ------------------------------------------
# CHẶNG 1: SELF-ATTENTION & ĐƯỜNG TẮT
# ------------------------------------------
x_goc_1 = x # Lưu lại bản sao của dữ liệu gốc

x_chuan_hoa_1 = ln1(x) # Qua trạm chuẩn hóa 1
print("x chuẩn hóa 1\n",x_chuan_hoa_1)

# Qua trạm Attention (Hàm này trả về kết quả và trọng số, ta lấy [0] là kết quả)
x_sau_attention = attention(x_chuan_hoa_1, x_chuan_hoa_1, x_chuan_hoa_1)[0]
print("x sau khi qua lớp attention\n", x_sau_attention)

# 🚀 MA THUẬT RESIDUAL CONNECTION 1:
# Lấy kết quả sau Attention CỘNG NGƯỢC với dữ liệu gốc!
x = x_goc_1 + x_sau_attention 
print("Lấy kết quả sau Attention CỘNG NGƯỢC với dữ liệu gốc!\n", x)


# ------------------------------------------
# CHẶNG 2: FEED-FORWARD & ĐƯỜNG TẮT
# ------------------------------------------
x_goc_2 = x # Lưu lại bản sao của x (lúc này x đã chứa thông tin Attention)

x_chuan_hoa_2 = ln2(x) # Qua trạm chuẩn hóa 2
print("tiếp tục đưa qua lớp chuẩn hóa thứ 2\n", x_chuan_hoa_2)

x_sau_ffn = ffn(x_chuan_hoa_2) # Qua trạm Suy nghĩ sâu
print("qua lớp deep learning\n", x_sau_ffn)

# 🚀 MA THUẬT RESIDUAL CONNECTION 2:
# Lấy kết quả sau FFN CỘNG NGƯỢC với x_goc_2!
x = x_goc_2 + x_sau_ffn
print("Lấy kết quả sau FFN CỘNG NGƯỢC với x_goc_2!\n",x)


# ==========================================
# KIỂM TRA KẾT QUẢ CUỐI CÙNG
# ==========================================
print("\n2. Hình dáng Đầu ra sau toàn bộ chặng đường:", x.shape)
print("=> Kích thước [1, 3, 4] được giữ nguyên hoàn hảo! Dữ liệu đã giàu ngữ cảnh hơn rất nhiều.")

--- CHƯƠNG 6: DÒNG CHẢY QUA KHỐI TRANSFORMER ---
1. Hình dáng Đầu vào ban đầu: torch.Size([1, 3, 4])
x chuẩn hóa 1
 tensor([[[ 0.7086,  1.2057, -1.2827, -0.6316],
         [-0.1402,  1.1413, -1.5437,  0.5427],
         [ 0.5376, -0.3305, -1.4415,  1.2344]]],
       grad_fn=<NativeLayerNormBackward0>)
x sau khi qua lớp attention
 tensor([[[ 0.5016, -0.3930, -0.5673, -0.5153],
         [ 0.4949, -0.3927, -0.5828, -0.5181],
         [ 0.4809, -0.4014, -0.6455, -0.5401]]], grad_fn=<TransposeBackward0>)
Lấy kết quả sau Attention CỘNG NGƯỢC với dữ liệu gốc!
 tensor([[[ 1.0264e+00,  5.7847e-01, -1.8321e+00, -1.1950e+00],
         [ 9.1037e-01,  6.4725e-01, -8.5141e-01,  2.3016e-01],
         [ 1.5216e-03, -1.8493e+00, -3.3331e+00, -2.4193e-01]]],
       grad_fn=<AddBackward0>)
tiếp tục đưa qua lớp chuẩn hóa thứ 2
 tensor([[[ 1.1610,  0.7847, -1.2405, -0.7053],
         [ 1.0063,  0.6148, -1.6153, -0.0059],
         [ 1.0091, -0.3670, -1.4702,  0.8281]]],
       grad_fn=<NativeLayerNormBackwar

---

# 🎯 CHƯƠNG 7: LỚP ĐẦU RA - DỰ ĐOÁN TỪ TIẾP THEO (OUTPUT LAYER)

---
Sau khi đi qua N vòng lặp Transformer, vector của mỗi từ đã đạt đến đỉnh cao của sự thông minh. Nhưng nó vẫn đang mang kích thước `d_model` (Vd: 8 chiều).
Nhiệm vụ cuối cùng: Biến vector 8 chiều này thành phần trăm xác suất dự đoán cho TẤT CẢ các từ trong từ điển.

### 1. Phóng to về kích thước Từ Điển (Linear Projection)
* Ta dùng một lớp `nn.Linear(d_model, vocab_size)`.
* Ví dụ: Từ điển có 5000 từ. Nó sẽ biến vector 8 chiều thành vector 5000 chiều. Mỗi con số đại diện cho "điểm tự tin" của một từ trong từ điển.

### 2. Biến điểm số thành Xác Suất (Softmax)
* Đưa 5000 điểm số đó qua hàm Softmax để tổng của chúng bằng 1.0 (100%).
* Từ nào có % cao nhất chính là từ mà AI quyết định sẽ "nhả" ra màn hình!

In [9]:
import torch
import torch.nn as nn

print("--- CHƯƠNG 7: TỪ BỘ NÃO RA MIỆNG CỦA AI ---")

d_model = 8
vocab_size = 5000 # Giả sử từ điển của ta có 5000 từ

# Giả sử x là kết quả trò thu được sau vòng lặp 6 lần ở Chương 6
# (1 câu, 5 từ, mỗi từ 8 chiều)
x_final = torch.randn(1, 5, d_model)

# ==========================================
# 1. KHỞI TẠO LỚP ĐẦU RA (OUTPUT LAYER)
# ==========================================
# Chuẩn hóa lần cuối cho an toàn
ln_final = nn.LayerNorm(d_model)

# Lớp tuyến tính ánh xạ từ d_model (8) sang toàn bộ từ điển (5000)
# Người ta hay gọi lớp này là "lm_head" (Language Model Head)
lm_head = nn.Linear(in_features=d_model, out_features=vocab_size)

# ==========================================
# 2. CHẠY DỮ LIỆU ĐỂ DỰ ĐOÁN
# ==========================================
x_chuan_hoa = ln_final(x_final)

# Đưa ra điểm số dự đoán cho toàn bộ từ điển (gọi là Logits)
logits = lm_head(x_chuan_hoa)

print("1. Hình dáng kết quả (Logits):", logits.shape)
# Output sẽ là [1, 5, 5000] (1 câu, 5 từ, mỗi từ đưa ra 5000 điểm dự đoán)

# ------------------------------------------
# LẤY TỪ DỰ ĐOÁN CUỐI CÙNG (CỦA TỪ THỨ 5)
# ------------------------------------------
# Ta chỉ quan tâm đến việc dự đoán từ TIẾP THEO (sau từ thứ 5)
diem_so_tu_cuoi = logits[0, -1, :] # Lấy từ cuối cùng (-1)

# Ép thành phần trăm %
xac_suat = torch.softmax(diem_so_tu_cuoi, dim=-1)

# Tìm ID của từ có phần trăm cao nhất (argmax)
tu_duoc_chon = torch.argmax(xac_suat).item()
phan_tram_tu_tin = xac_suat[tu_duoc_chon].item() * 100

print(f"\n2. KẾT LUẬN CỦA AI:")
print(f"🔮 AI dự đoán từ tiếp theo sẽ là từ có ID số: {tu_duoc_chon}")
print(f"📊 Độ tự tin: {phan_tram_tu_tin:.2f}% (Vì chưa Train nên nó đang đoán bừa)")

--- CHƯƠNG 7: TỪ BỘ NÃO RA MIỆNG CỦA AI ---
1. Hình dáng kết quả (Logits): torch.Size([1, 5, 5000])

2. KẾT LUẬN CỦA AI:
🔮 AI dự đoán từ tiếp theo sẽ là từ có ID số: 3339
📊 Độ tự tin: 0.10% (Vì chưa Train nên nó đang đoán bừa)
